# Video Merging

This tool merges exported videos from `4_Video_Export.ipynb` into a single video of all imaging channels combined. It used `videos_grid.py`, which should be included in the same directory

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import warnings
warnings.filterwarnings(action='once',category=UserWarning)

import videos_grid
from IPython.display import Video

import os

# addition of active memory manager
import dask
dask.config.set({'distributed.scheduler.active-memory-manager.start': True});
dask.config.set({'distributed.scheduler.worker-ttl': "5m"});
dask.config.set({'distributed.scheduler.allowed-failures': 100});

In [ ]:
headpath_videos = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/AVI_Library_MP4"
output_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/AVI_Library_MP4/Merged/"
channels = ["mCherry", "YFP"]

In [ ]:
mch_files = os.listdir(headpath_videos+'/'+channels[0])
yfp_files = os.listdir(headpath_videos+'/'+channels[1])

mch_ids = sorted([file.split('.')[0] for file in mch_files])
yfp_ids = sorted([file.split('.')[0] for file in yfp_files])

mch_ids == yfp_ids

In [ ]:
# Function to merge videos
def merge_video(oDEPool7_id,
                     headpath_videos = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/AVI_Library_MP4",
                     output_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/AVI_Library_MP4/Merged/",
                     channels = ["mCherry", "YFP"]):
    videos_to_merge = [headpath_videos + '/' + channel + '/' + oDEPool7_id + '.mp4' for channel in channels]
    videos_grid.merge_videos(
    videos_to_merge,
    output_path+ oDEPool7_id+'.mp4',
    grid_size=(2, 1),  # 2 row, 2 cols
    max_frames=201)  # merge first 100 frames per video
    return oDEPool7_id

## Run in Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="2:00:00",
    local=False,
    n_workers=50,
    n_workers_min=50,
    memory="4GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()
dask_controller.displaydashboard()

In [ ]:
# Run in Dask
tr.writedir(output_path, overwrite=True)
futures_dict = {}
for oDEPool7_id in mch_ids:
    futures_dict['oDEPool7_id: '+str(oDEPool7_id)] = dask_controller.daskclient.submit(merge_video, oDEPool7_id)

## Run locally

In [ ]:
# Run locally
for oDEPool7_id in mch_ids[:10]:
    videos_to_merge = [headpath_videos + '/' + channel + '/' + oDEPool7_id + '.mp4' for channel in channels]
    videos_grid.merge_videos(
    videos_to_merge,
    output_path+ oDEPool7_id+'.mp4',
    grid_size=(2, 1),  # 2 row, 2 cols
    max_frames=201)  # merge first 100 frames per video